In [9]:
%%capture
%pip install -U pageindex google-genai python-dotenv

In [10]:
from dotenv import load_dotenv
import os
load_dotenv()
PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")  
print("Loaded PageIndex key", "Yes" if PAGEINDEX_API_KEY else "No")
print("Loaded gemini key", "Yes" if GOOGLE_API_KEY else "No")

Loaded PageIndex key Yes
Loaded gemini key Yes


In [15]:
# just testing it
from google import genai
from pageindex import PageIndexClient

gemini_client = genai.Client(api_key=GOOGLE_API_KEY)
pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)

Upload and Index PDF


In [17]:
PDF_PATH = r"C:\Users\saksh\Downloads\B.Tech. CS_CE and CSE Syllabus  3rd Year 2024-25 (1).pdf"
result = pi_client.submit_document(PDF_PATH)
doc_id = result['doc_id']
print(f"The document id is: {doc_id}")


The document id is: pi-cmnmarv8r0u6w01qpilgh39ww


In [20]:
# building tree index
import time
while True:
    status_result = pi_client.get_document(doc_id)
    status = status_result.get('status')
    print(f"Status: {status}")
    if status == "completed":
        print("\n Tree index ready!")
        break
    elif status == "failed":
        print("\n Processing failed. Check your PDF format.")
        break
    
    time.sleep(5)


Status: completed

 Tree index ready!


In [21]:
# fetch how first node looks  
import json
tree_result = pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_result.get('result', [])
print(f" Top-level sections: {len(pageindex_tree)}")
print("\n Raw tree (first node):")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

 Top-level sections: 2

 Raw tree (first node):
{
  "title": "DR. A.P.J. ABDUL KALAM TECHNICAL UNIVERSITY, UTTAR PRADESH, LUCKNOW",
  "node_id": "0000",
  "page_index": 1,
  "summary": "# DR. A.P.J. ABDUL KALAM TECHNICAL UNIVERSITY, UTTAR PRADESH, LUCKNOW\n\n![img-0.jpeg](img-0.jpeg)\n",
  "text": "# DR. A.P.J. ABDUL KALAM TECHNICAL UNIVERSITY, UTTAR PRADESH, LUCKNOW\n\n![img-0.jpeg](img-0.jpeg)\n"
}


In [22]:
# ── Pretty-print the full tree ───────────────────────────────────────────────
def print_tree(nodes, indent=0):
    """Recursively print tree titles for a visual overview."""
    for node in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        page   = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']}  (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

print(" Full Document Structure:\n")
print_tree(pageindex_tree)

 Full Document Structure:

[0000] DR. A.P.J. ABDUL KALAM TECHNICAL UNIVERSITY, UTTAR PRADESH, LUCKNOW  (p.1)
[0001] EVALUATION SCHEME &amp; SYLLABUS FOR B. TECH. THIRD YEAR  (p.1)
  └─ [0002] Departmental Elective-I  (p.3)
  └─ [0003] Departmental Elective-II  (p.3)
  └─ [0004] Departmental Elective-III  (p.3)
    └─ [0005] B.Tech CS/CSE Curriculum: Semesters V & VI - Core Subjects, Electives, and Detailed Syllabi  (p.3)
    └─ [0006] Design and Analysis of Algorithm (BCS503)  (p.7)
    └─ [0007] Departmental Elective-III: Course Details and Syllabi  (p.11)
    └─ [0008] Departmental Elective-III: Course Syllabi and Lab Details  (p.15)
  └─ [0009] Database Management Systems Lab (BCS-551): Mapping with Virtual Lab  (p.19)
    └─ [0010] Lab Syllabi: Database Management, Web Technologies, Algorithms, and Software Engineering  (p.19)
    └─ [0011] Curriculum and Syllabi for V & VI Semesters: Compiler Design, Computer Networks, and Big Data  (p.23)
    └─ [0012] Big Data Technologies: Hive

In [23]:
# ── Count total nodes ────────────────────────────────────────────────────────
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

total = count_nodes(pageindex_tree)
print(f" Total nodes in tree: {total}") #each node is 1 retrieval section


 Total nodes in tree: 15


## LLM Tree Search
PageIndex retrieval:
query + tree → LLM reasons → "node 0007 and 0008 contain the answer"

In [66]:
# ── LLM Tree Search Function ─────────────────────────────────────────────────
import json
def llm_tree_search(query: str, tree: list, model: str = "gemini-2.5-flash") -> dict:
    """
    Core PageIndex retrieval:
    Sends the query + document tree to an LLM.
    LLM reasons over the structure and returns relevant node_ids.
    
    Returns: dict with 'thinking' (reasoning) and 'node_list' (node IDs)
    """
    
    # Compress tree to save tokens — only send titles + short summaries
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title":   n["title"],
                "page":    n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]  # first 150 chars
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out
    
    compressed_tree = compress(tree)
    
    prompt = f"""You are given a query and a document's tree structure (like a Table of Contents).
Your task: identify which node IDs most likely contain the answer to the query.
Think step-by-step about which sections are relevant.

Query: {query}

Document Tree:
{json.dumps(compressed_tree, indent=2)}

Reply ONLY in this exact JSON format:
{{
  "thinking": "<your step-by-step reasoning>",
  "node_list": ["node_id1", "node_id2"]
}}"""

    response = gemini_client.models.generate_content(
    model=model,
    contents=prompt
)

    try:
        result_dict = json.loads(response.text)
    except json.JSONDecodeError:
        result_dict = {"thinking": "Error parsing response", "node_list": []}

    return result_dict

In [67]:
# ── Test with a sample query ─────────────────────────────────────────────────
query = "What is the syllabus covered in DBMS open elective?"

print(f" Query: {query}\n")
result = llm_tree_search(query, pageindex_tree)

print(" LLM Reasoning:")
print(result.get("thinking", "N/A"))
print()
print(" Selected Node IDs:", result.get("node_list", []))

 Query: What is the syllabus covered in DBMS open elective?



ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 19.753884696s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flash'},

##End-to-End RAG pipeline

3 steps:

Tree Search → LLM picks relevant node_ids

Retrieve → Fetch the actual section content from those nodes

Generate → LLM writes a grounded answer with page citations

In [ ]:
# ── Helper: Find nodes by ID ─────────────────────────────────────────────────

def find_nodes_by_ids(tree: list, target_ids: list) -> list:
    """Recursively walk the tree and collect nodes matching target_ids."""
    found = []
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return found

In [ ]:
# ── Generate answer from retrieved nodes ─────────────────────────────────────

def generate_answer(query: str, nodes: list, model: str = "gemini-2.5-flash") -> str:
    """
    Takes retrieved nodes as context and generates a grounded answer.
    Instructs the LLM to cite section titles and page numbers.
    """
    if not nodes:
        return "No relevant sections found in the document."
    
    # Build context string from retrieved nodes
    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' | Page {node.get('page_index', '?')}]\n"
            f"{node.get('text', 'Content not available.')}"
        )
    context = "\n\n---\n\n".join(context_parts)
    
    prompt = f"""You are an expert document analyst.
Answer the question using ONLY the provided context.
For every claim you make, cite the section title and page number in parentheses.
Be concise and precise.

Question: {query}

Context:
{context}

Answer:"""
    
    response = gemini_client.models.generate_content(
    model=model,
    contents=prompt
)
    
    return response.candidates[0].content.parts[0].text

In [ ]:
# ── The complete Vectorless RAG function ─────────────────────────────────────
# THE COMPLETE PIPELINE TO THE ENTIRE RAG
# Find the right sections -> Get the text for those sections -> Read the text to answer the question.

def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    """
    Full end-to-end PageIndex RAG pipeline:
    
    Step 1: LLM Tree Search  → finds relevant node_ids
    Step 2: Node Retrieval   → fetches section content
    Step 3: Answer Generation → produces cited answer
    """
    if verbose:
        print(f"{'='*55}")
        print(f" Query: {query}")
        print(f"{'='*55}")
    
    # Step 1: Tree Search
    search_result  = llm_tree_search(query, tree)
    node_ids = search_result.get("node_list", [])
    
    if verbose:
        print(f"\n Reasoning: {search_result.get('thinking', '')[:200]}...")
        print(f" Retrieved node IDs: {node_ids}")
    
    # Step 2: Retrieve nodes
    nodes = find_nodes_by_ids(tree, node_ids)
    
    if verbose:
        print(f" Sections found: {[n['title'] for n in nodes]}")
    
    # Step 3: Generate answer
    answer = generate_answer(query, nodes)
    
    if verbose:
        print(f"\n Answer:\n{answer}")
    
    return answer

In [ ]:
# ── Run the full pipeline ────────────────────────────────────────────────────
answer = vectorless_rag(
    query="What are the syllabus covered in DBMS ?",
    tree=pageindex_tree
)

 Query: What are the syllabus covered in DBMS ?

 Reasoning: The user is asking for the syllabus related to DBMS. I need to find nodes that explicitly mention 'syllabus' and 'DBMS' or 'Database Management System'.

1.  **'EVALUATION SCHEME &amp; SYLLABUS FOR B....
 Retrieved node IDs: ['0010', '0013']
 Sections found: ['Lab Syllabi: Database Management, Web Technologies, Algorithms, and Software Engineering', 'Syllabus Overview: Data Compression, Software Engineering, Compiler Design, Computer Networks Labs, and Database Management System Open Elective']

 Answer:
The syllabus covered in DBMS includes both theoretical concepts and practical applications.

For the Database Management Systems Lab (BCS-551), the syllabus involves:
*   Data Definition Language (DDL) Statements: Create table, Alter table, Drop table (Lab Syllabi: Database Management, Web Technologies, Algorithms, and Software Engineering, Page 19)
*   Data Manipulation Language (DML) Statements (Lab Syllabi: Database Manage

## Summary
1) llm_tree_search() — LLM reasons over document tree to find relevant nodes
2) find_nodes_by_ids() — Retrieve actual section content from tree
3) generate_answer() — LLM produces cited, grounded answers
4) vectorless_rag() — Full pipeline combining all 3 steps
5) expert_rag() — Domain-guided retrieval without any fine-tuning